# DPA - Volume (Part 1): Network Traffic Analysis

Our research questions:
1. What traffic patterns distinguish normal activity from attacks?
2. Which network services and traffic characteristics are associated with different attack categories?

This notebook currently contains:
- Optional download of the two source datasets.
- Step 1: inspect the actual files and small traffic samples.
- Step 2: define explicit schemas and check column names.
- Step 3: test small-sample CSV ingestion and Parquet write/read.

Run the cells from top to bottom. Leave `DOWNLOAD_DATA = False` when the datasets are already available. Step 1 uses ordinary Python. Step 2 requires PySpark installed in the current kernel. Step 3 starts Spark and also requires Java; the configured Docker environment provides both.


## Optional ? Download the datasets

Leave `DOWNLOAD_DATA = False` when the files are already available. Set it to `True` to download the original traffic CSVs and the IANA registry. Existing nonempty files are skipped. Downloads use ordinary Python and do not require Spark or Docker.

**Part 1 uses `data/CSECICIDS2018_original/` and the IANA registry.** The original traffic CSVs total about 6.89 GB. Interrupted downloads restart from the beginning when rerun.


In [1]:
DOWNLOAD_DATA = False  # Optional: change to True to download missing source files.

from pathlib import Path

# Locate the project from notebooks/, the project root, or the outer workspace.
current_folder = Path.cwd().resolve()
root_candidates = [current_folder, *current_folder.parents]
ROOT = next((project for folder in root_candidates
             for project in (folder, folder / "DPA-Volume-Part1-Project")
             if (project / "downloaders" / "download_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder or its notebooks/ folder.")
print("Project folder:", ROOT)

if DOWNLOAD_DATA:
    import runpy
    # Load the downloader functions, then download the datasets.
    downloader = runpy.run_path(str(ROOT / "downloaders" / "download_data.py"))
    downloader["prepare_iana"](ROOT / "data")
    downloader["prepare_traffic"](ROOT / "data")
    print("Source datasets are ready. No Spark processing was performed.")
else:
    print("Download skipped. Set DOWNLOAD_DATA = True and rerun this cell if needed.")

Project folder: /home/jovyan/work
Download skipped. Set DOWNLOAD_DATA = True and rerun this cell if needed.


## Step 1 - Inspect the actual source files (no Spark required)

**Purpose:** understand the raw columns and values before deciding their Spark types or cleaning rules. Run the optional download cell above first, with `DOWNLOAD_DATA = False` when files are already present.

We check every traffic header and read just **100 records at each of five positions per file** (start, 25%, 50%, 75%, 99%). This helps include later attack records without scanning gigabytes. These are systematic inspection windows, **not a random or representative statistical sample**. Do not use their counts to estimate attack frequencies. The traffic format is expected to have one CSV record per physical line; unexpected record widths cause an error for review.

The IANA CSV is small, so we inspect its complete registry to find port ranges and repeated keys. All fields remain raw strings. No records are cleaned, relabelled, or removed.

In [2]:
import csv
import io
import json
from collections import Counter
from datetime import datetime
from itertools import islice

traffic_folder = ROOT / "data" / "CSECICIDS2018_original"
traffic_files = sorted(traffic_folder.glob("*.csv"))
iana_folder = ROOT / "data" / "Service and port name"
iana_files = sorted(iana_folder.glob("service-names-port-numbers*.csv"))
assert traffic_files, f"No traffic CSVs found in {traffic_folder}"
assert iana_files, f"No IANA registry found in {iana_folder}"
# Prefer the standard filename, while accepting the existing '(3)' filename.
registry_file = next((p for p in iana_files if p.name == "service-names-port-numbers.csv"), iana_files[0])

print("Traffic CSVs:", len(traffic_files))
for path in traffic_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**3:.2f} GiB")
print("IANA:", registry_file.name)

Traffic CSVs: 10
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 3.78 GiB
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 0.10 GiB
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 0.35 GiB
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 0.36 GiB
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 0.33 GiB
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 0.31 GiB
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 0.19 GiB
IANA: service-names-port-numbers (3).csv


### 1A - Compare the headers

We compare names, order, and duplicate column names. Nine original files have 80 columns. The 20 February file has four extra fields at the beginning: `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. We define a schema for each layout in Step 2. Source names such as `Tot Fwd Pkts` are kept as supplied.


In [3]:
traffic_headers = {}
header_summary = []
for path in traffic_files:
    with path.open(encoding="utf-8-sig", newline="") as source:
        header = next(csv.reader(source))
    assert len(header) == len(set(header)), f"Duplicate column names in {path.name}"
    traffic_headers[path.name] = header
    reference_header = traffic_headers[traffic_files[0].name]
    header_summary.append({
        "file": path.name,
        "size_bytes": path.stat().st_size,
        "column_count": len(header),
        "matches_first_header": header == reference_header,
    })
    print(f"{path.name}: {len(header)} columns; same names/order = {header == reference_header}")

print("\nColumns in the first traffic file:")
for position, name in enumerate(reference_header, start=1):
    print(f"{position:2}. {name}")

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 84 columns; same names/order = False
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 80 columns; same names/order = True

Columns in the first traffic file:
 1. Dst Port
 2. Protocol
 3. Timestamp
 4. Flow Duration
 5. Tot Fwd Pkts
 6. Tot Bwd Pkts
 7. 

### 1B - Inspect small windows of real traffic

`seek` jumps to a byte position without reading the intervening data. We skip the header or partial physical line, then read at most 100 records. Every sampled record must match its file's header width. The original files are opened for reading only.

Keep samples grouped by file because the original CSVs have two header layouts. The windows help explore different parts of the files, but do not guarantee coverage of every attack.


In [4]:
ROWS_PER_WINDOW = 100
WINDOW_POSITIONS = (0.0, 0.25, 0.5, 0.75, 0.99)

def inspect_window(path, header, fraction):
    with path.open("rb") as source:
        source.seek(int(path.stat().st_size * fraction))
        source.readline()  # Skip the header, or the partial line at this position.
        with io.TextIOWrapper(source, encoding="utf-8", newline="") as text:
            records = list(islice(csv.reader(text), ROWS_PER_WINDOW))
    assert all(len(row) == len(header) for row in records), f"Unexpected CSV record width in {path.name}"
    return [dict(zip(header, row)) for row in records]

sample_rows = []
sample_rows_by_file = {}
window_summary = []
for path in traffic_files:
    sample_rows_by_file[path.name] = []
    for fraction in WINDOW_POSITIONS:
        rows = inspect_window(path, traffic_headers[path.name], fraction)
        sample_rows.extend(rows)
        sample_rows_by_file[path.name].extend(rows)
        window_summary.append({"file": path.name, "position_fraction": fraction, "sampled_rows": len(rows)})
    print(f"{path.name}: inspected {sum(w['sampled_rows'] for w in window_summary if w['file'] == path.name)} records")
print("Total inspected traffic records:", len(sample_rows))

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records


Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: inspected 500 records
Total inspected traffic records: 5000


### 1C - Examine labels, protocols, timestamps, and measurements

These counts describe **only the inspected windows**. Preserve the original attack labels, including their spelling. Numeric protocol codes will need matching to IANA's text protocols before joining.

Original timestamps use `day/month/year hour:minute:second`; we also check fractional seconds. Keep the source strings for now. They have no timezone offset. Negative initial TCP window values such as `-1` need interpretation before cleaning.


In [5]:
sample_labels = Counter(row["Label"] for row in sample_rows)
sample_protocols = Counter(row["Protocol"] for row in sample_rows)
print("Labels in inspected windows:", dict(sample_labels))
print("Raw protocol codes:", dict(sample_protocols))

formats = ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M:%S.%f")
format_counts = Counter()
parsed_times = []
unparsed_times = []
for row in sample_rows:
    for timestamp_format in formats:
        try:
            parsed_times.append(datetime.strptime(row["Timestamp"], timestamp_format))
            format_counts[timestamp_format] += 1
            break
        except ValueError:
            continue
    else:
        unparsed_times.append(row["Timestamp"])
print("Timestamp formats matched:", dict(format_counts))
print("Unparsed sampled timestamps:", len(unparsed_times))
if parsed_times:
    print("Sample timestamp range:", min(parsed_times), "to", max(parsed_times))

example_columns = ["Dst Port", "Protocol", "Timestamp", "Flow Duration",
                   "Tot Fwd Pkts", "Tot Bwd Pkts", "Label"]
examples = {}
for row in sample_rows:
    if row["Label"] not in examples:
        examples[row["Label"]] = {column: row[column] for column in example_columns}
print("\nOne raw example per observed label:")
print(json.dumps(examples, indent=2))

sentinel_examples = {}
for column in ("Init Fwd Win Byts", "Init Bwd Win Byts"):
    sentinel_examples[column] = sorted({row[column] for row in sample_rows if row[column].startswith("-")})
print("Observed negative values requiring interpretation:", sentinel_examples)


Labels in inspected windows: {'Benign': 4140, 'DoS attacks-SlowHTTPTest': 103, 'DoS attacks-Hulk': 164, 'DDoS attacks-LOIC-HTTP': 37, 'Infilteration': 144, 'DoS attacks-GoldenEye': 56, 'FTP-BruteForce': 6, 'SSH-Bruteforce': 99, 'DDOS attack-HOIC': 251}
Raw protocol codes: {'6': 3511, '0': 325, '17': 1164}


Timestamp formats matched: {'%d/%m/%Y %H:%M:%S': 5000}
Unparsed sampled timestamps: 0
Sample timestamp range: 2018-02-14 01:22:05 to 2018-03-02 12:57:21

One raw example per observed label:
{
  "Benign": {
    "Dst Port": "443",
    "Protocol": "6",
    "Timestamp": "02/03/2018 08:47:38",
    "Flow Duration": "141385",
    "Tot Fwd Pkts": "9",
    "Tot Bwd Pkts": "7",
    "Label": "Benign"
  },
  "DoS attacks-SlowHTTPTest": {
    "Dst Port": "21",
    "Protocol": "6",
    "Timestamp": "16/02/2018 10:12:14",
    "Flow Duration": "21",
    "Tot Fwd Pkts": "1",
    "Tot Bwd Pkts": "1",
    "Label": "DoS attacks-SlowHTTPTest"
  },
  "DoS attacks-Hulk": {
    "Dst Port": "80",
    "Protocol": "6",
    "Timestamp": "16/02/2018 01:46:06",
    "Flow Duration": "20666",
    "Tot Fwd Pkts": "2",
    "Tot Bwd Pkts": "0",
    "Label": "DoS attacks-Hulk"
  },
  "DDoS attacks-LOIC-HTTP": {
    "Dst Port": "80",
    "Protocol": "6",
    "Timestamp": "20/02/2018 10:13:54",
    "Flow Duration": "3904",

### 1D - Inspect the IANA registry before choosing the join

This secondary dataset is small enough to inspect with Python's CSV reader. Some port values represent ranges, and some rows have no service or transport protocol. Multiple entries can share a literal `(Port Number, Transport Protocol)` key.

We count these situations here; we do not expand ranges, drop rows, choose a service, or join yet. A registered service is an association, not proof of the actual application that produced a flow.

In [6]:
with registry_file.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    registry_header = reader.fieldnames
    registry_rows = list(reader)  # Only the small IANA registry, never the traffic files.
print("IANA columns:", registry_header)
assert len(registry_header) == len(set(registry_header)), "Duplicate IANA column names"
assert {"Port Number", "Transport Protocol", "Service Name"}.issubset(registry_header)
assert all(None not in row and None not in row.values() for row in registry_rows), "Unexpected IANA record width"

registry_keys = Counter((row["Port Number"], row["Transport Protocol"]) for row in registry_rows)
range_ports = [row["Port Number"] for row in registry_rows if "-" in row["Port Number"]]
iana_summary = {
    "file": registry_file.name,
    "row_count": len(registry_rows),
    "column_count": len(registry_header),
    "transport_protocol_counts": dict(Counter(row["Transport Protocol"] for row in registry_rows)),
    "rows_with_port_ranges": len(range_ports),
    "port_range_examples": list(dict.fromkeys(range_ports))[:5],
    "rows_with_blank_port": sum(not row["Port Number"].strip() for row in registry_rows),
    "rows_with_blank_protocol": sum(not row["Transport Protocol"].strip() for row in registry_rows),
    "rows_with_blank_service": sum(not row["Service Name"].strip() for row in registry_rows),
    "repeated_literal_key_count": sum(count > 1 for count in registry_keys.values()),
    "extra_rows_on_repeated_literal_keys": sum(count - 1 for count in registry_keys.values()),
}
print(json.dumps(iana_summary, indent=2))
print("Example registered service:", next(row for row in registry_rows if row["Service Name"] and row["Port Number"].isdigit()))

IANA columns: ['Service Name', 'Port Number', 'Transport Protocol', 'Description', 'Assignee', 'Contact', 'Registration Date', 'Modification Date', 'Reference', 'Service Code', 'Unauthorized Use Reported', 'Assignment Notes']
{
  "file": "service-names-port-numbers (3).csv",
  "row_count": 14536,
  "column_count": 12,
  "transport_protocol_counts": {
    "tcp": 6608,
    "udp": 6357,
    "sctp": 93,
    "dccp": 11,
    "": 1467
  },
  "rows_with_port_ranges": 766,
  "port_range_examples": [
    "225-241",
    "249-255",
    "272-279",
    "288-299",
    "301-307"
  ],
  "rows_with_blank_port": 1073,
  "rows_with_blank_protocol": 1467,
  "rows_with_blank_service": 1724,
  "repeated_literal_key_count": 230,
  "extra_rows_on_repeated_literal_keys": 1300
}
Example registered service: {'Service Name': 'tcpmux', 'Port Number': '1', 'Transport Protocol': 'tcp', 'Description': 'TCP Port Service Multiplexer', 'Assignee': '[Mark_Lottor]', 'Contact': '[Mark_Lottor]', 'Registration Date': '', 'Mod

### 1E - Save the inspection evidence

The JSON stores header checks, the sampling method, observed values, and registry counts. The CSV stores one header-summary row per traffic file. These are inspection reports, not a cleaning audit or final research results. No source data is changed.

In [7]:
inspection_folder = ROOT / "reports" / "inspection" / "original"
inspection_folder.mkdir(parents=True, exist_ok=True)
inspection_summary = {
    "scope": "Source inspection only; no cleaning, typed ingestion, or Spark analysis performed",
    "traffic": {
        "source_folder": str(traffic_folder.relative_to(ROOT)),
        "file_count": len(traffic_files),
        "total_file_bytes": sum(path.stat().st_size for path in traffic_files),
        "header_summary": header_summary,
        "headers_by_file": traffic_headers,
        "sampling_method": "100 records after each byte position: 0%, 25%, 50%, 75%, 99%; not statistically representative",
        "window_summary": window_summary,
        "sample_row_count": len(sample_rows),
        "sample_label_counts": dict(sample_labels),
        "sample_protocol_counts": dict(sample_protocols),
        "sample_timestamp_formats": dict(format_counts),
        "sample_timestamp_min": min(parsed_times).isoformat() if parsed_times else None,
        "sample_timestamp_max": max(parsed_times).isoformat() if parsed_times else None,
        "sample_unparsed_timestamp_count": len(unparsed_times),
        "sample_negative_sentinel_values": sentinel_examples,
        "raw_examples_by_observed_label": examples,
    },
    "iana": iana_summary,
}
summary_file = inspection_folder / "inspection_summary.json"
summary_file.write_text(json.dumps(inspection_summary, indent=2), encoding="utf-8")
with (inspection_folder / "traffic_headers.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(header_summary[0]))
    writer.writeheader()
    writer.writerows(header_summary)
print("Saved:", summary_file)
print("Saved:", inspection_folder / "traffic_headers.csv")

Saved: /home/jovyan/work/reports/inspection/original/inspection_summary.json
Saved: /home/jovyan/work/reports/inspection/original/traffic_headers.csv


**Step 1 complete.** Continue to Step 2 below to define and validate the explicit source schemas.


## Step 2 - Define the column types

A schema gives Spark the name and type of each CSV column. We define these types explicitly:

- `StringType`: labels, raw timestamps, and optional flow/IP identifiers.
- `IntegerType`: ports and protocol codes.
- `LongType`: whole-number packet counts, flags, header lengths, and flow durations.
- `DoubleType`: averages, rates, ratios, and measurements written with decimals or scientific notation.

For example, some byte totals use `0.0` and IAT measurements use `1.19E+08`, so those fields use doubles. `True` allows missing values so we can count and handle them later. Timestamps remain text until cleaning.

We use an 80-column schema for nine files and an 84-column schema for 20 February.


In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

TRAFFIC_SCHEMA = StructType([
    StructField("Dst Port", IntegerType(), True),
    StructField("Protocol", IntegerType(), True),
    StructField("Timestamp", StringType(), True),
    StructField("Flow Duration", LongType(), True),
    StructField("Tot Fwd Pkts", LongType(), True),
    StructField("Tot Bwd Pkts", LongType(), True),
    StructField("TotLen Fwd Pkts", DoubleType(), True),
    StructField("TotLen Bwd Pkts", DoubleType(), True),
    StructField("Fwd Pkt Len Max", DoubleType(), True),
    StructField("Fwd Pkt Len Min", DoubleType(), True),
    StructField("Fwd Pkt Len Mean", DoubleType(), True),
    StructField("Fwd Pkt Len Std", DoubleType(), True),
    StructField("Bwd Pkt Len Max", DoubleType(), True),
    StructField("Bwd Pkt Len Min", DoubleType(), True),
    StructField("Bwd Pkt Len Mean", DoubleType(), True),
    StructField("Bwd Pkt Len Std", DoubleType(), True),
    StructField("Flow Byts/s", DoubleType(), True),
    StructField("Flow Pkts/s", DoubleType(), True),
    StructField("Flow IAT Mean", DoubleType(), True),
    StructField("Flow IAT Std", DoubleType(), True),
    StructField("Flow IAT Max", DoubleType(), True),
    StructField("Flow IAT Min", DoubleType(), True),
    StructField("Fwd IAT Tot", DoubleType(), True),
    StructField("Fwd IAT Mean", DoubleType(), True),
    StructField("Fwd IAT Std", DoubleType(), True),
    StructField("Fwd IAT Max", DoubleType(), True),
    StructField("Fwd IAT Min", DoubleType(), True),
    StructField("Bwd IAT Tot", DoubleType(), True),
    StructField("Bwd IAT Mean", DoubleType(), True),
    StructField("Bwd IAT Std", DoubleType(), True),
    StructField("Bwd IAT Max", DoubleType(), True),
    StructField("Bwd IAT Min", DoubleType(), True),
    StructField("Fwd PSH Flags", LongType(), True),
    StructField("Bwd PSH Flags", LongType(), True),
    StructField("Fwd URG Flags", LongType(), True),
    StructField("Bwd URG Flags", LongType(), True),
    StructField("Fwd Header Len", LongType(), True),
    StructField("Bwd Header Len", LongType(), True),
    StructField("Fwd Pkts/s", DoubleType(), True),
    StructField("Bwd Pkts/s", DoubleType(), True),
    StructField("Pkt Len Min", DoubleType(), True),
    StructField("Pkt Len Max", DoubleType(), True),
    StructField("Pkt Len Mean", DoubleType(), True),
    StructField("Pkt Len Std", DoubleType(), True),
    StructField("Pkt Len Var", DoubleType(), True),
    StructField("FIN Flag Cnt", LongType(), True),
    StructField("SYN Flag Cnt", LongType(), True),
    StructField("RST Flag Cnt", LongType(), True),
    StructField("PSH Flag Cnt", LongType(), True),
    StructField("ACK Flag Cnt", LongType(), True),
    StructField("URG Flag Cnt", LongType(), True),
    StructField("CWE Flag Count", LongType(), True),
    StructField("ECE Flag Cnt", LongType(), True),
    StructField("Down/Up Ratio", DoubleType(), True),
    StructField("Pkt Size Avg", DoubleType(), True),
    StructField("Fwd Seg Size Avg", DoubleType(), True),
    StructField("Bwd Seg Size Avg", DoubleType(), True),
    StructField("Fwd Byts/b Avg", DoubleType(), True),
    StructField("Fwd Pkts/b Avg", DoubleType(), True),
    StructField("Fwd Blk Rate Avg", DoubleType(), True),
    StructField("Bwd Byts/b Avg", DoubleType(), True),
    StructField("Bwd Pkts/b Avg", DoubleType(), True),
    StructField("Bwd Blk Rate Avg", DoubleType(), True),
    StructField("Subflow Fwd Pkts", LongType(), True),
    StructField("Subflow Fwd Byts", LongType(), True),
    StructField("Subflow Bwd Pkts", LongType(), True),
    StructField("Subflow Bwd Byts", LongType(), True),
    StructField("Init Fwd Win Byts", LongType(), True),
    StructField("Init Bwd Win Byts", LongType(), True),
    StructField("Fwd Act Data Pkts", LongType(), True),
    StructField("Fwd Seg Size Min", LongType(), True),
    StructField("Active Mean", DoubleType(), True),
    StructField("Active Std", DoubleType(), True),
    StructField("Active Max", DoubleType(), True),
    StructField("Active Min", DoubleType(), True),
    StructField("Idle Mean", DoubleType(), True),
    StructField("Idle Std", DoubleType(), True),
    StructField("Idle Max", DoubleType(), True),
    StructField("Idle Min", DoubleType(), True),
    StructField("Label", StringType(), True),
])

TRAFFIC_WITH_IP_SCHEMA = StructType([
    StructField("Flow ID", StringType(), True),
    StructField("Src IP", StringType(), True),
    StructField("Src Port", IntegerType(), True),
    StructField("Dst IP", StringType(), True),
] + TRAFFIC_SCHEMA.fields)

print("Original traffic schemas:", len(TRAFFIC_SCHEMA), "and", len(TRAFFIC_WITH_IP_SCHEMA), "columns")


Original traffic schemas: 80 and 84 columns


### IANA schema

Keep IANA fields as text for now. Port values can contain ranges such as `225-241`, so we cannot read every port value as a single integer. Cleaning and conversion will come later.

In [9]:
IANA_SCHEMA = StructType([
    StructField("Service Name", StringType(), True),
    StructField("Port Number", StringType(), True),
    StructField("Transport Protocol", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Assignee", StringType(), True),
    StructField("Contact", StringType(), True),
    StructField("Registration Date", StringType(), True),
    StructField("Modification Date", StringType(), True),
    StructField("Reference", StringType(), True),
    StructField("Service Code", StringType(), True),
    StructField("Unauthorized Use Reported", StringType(), True),
    StructField("Assignment Notes", StringType(), True),
])

print("IANA schema:", len(IANA_SCHEMA), "columns")

IANA schema: 12 columns


### Check the column names and order

Select the matching explicit schema for each file, then check the entire header. Unexpected names or order stop execution. This checks structure only; Step 3 tests parsing values and writing Parquet.


In [10]:
traffic_schemas = {}
for file_name, header in traffic_headers.items():
    schema = TRAFFIC_WITH_IP_SCHEMA if "Flow ID" in header else TRAFFIC_SCHEMA
    assert header == schema.fieldNames(), f"Check the columns in {file_name}"
    traffic_schemas[file_name] = schema
    print(f"Header matches: {file_name} ({len(schema)} columns)")

assert registry_header == IANA_SCHEMA.fieldNames(), "Check the IANA columns"
print("IANA header matches.")


Header matches: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv (84 columns)
Header matches: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
Header matches: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (80 columns)
IANA header matches.


**Step 2 complete.** Both original traffic layouts and the IANA header match their explicit schemas. Continue to the small-sample ingestion test below.


## Step 3 - Test small-sample ingestion and Parquet

We will read CSV files with our explicit schemas, write typed Parquet, and read it back. Use only the 5,000 traffic records already inspected in Step 1, plus the small IANA registry. The original source files remain unchanged.

This step requires a working Spark/Java environment. Run the earlier cells first. The sample is for testing the pipeline, not measuring the full dataset's attack distribution.

### 3A - Save the traffic samples and start Spark

Save one small CSV per original source file under `data/intermediate/original/`. Each sample keeps its own header layout. These generated samples are replaced on rerun. Spark uses two local worker threads and four shuffle partitions.


In [11]:
sample_folder = ROOT / "data" / "intermediate" / "original"
sample_folder.mkdir(parents=True, exist_ok=True)
for path in traffic_files:
    with (sample_folder / path.name).open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=traffic_headers[path.name])
        writer.writeheader()
        writer.writerows(sample_rows_by_file[path.name])

from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]")
         .appName("DPA-Volume-Original-Sample")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Sample traffic records:", len(sample_rows))


Sample traffic records: 5000


### 3B - Read the CSVs using their schemas

Read each traffic sample with its matching explicit schema. `enforceSchema=False` checks header names, and `FAILFAST` stops on parsing errors. IANA also needs multiline quoted fields handled correctly.

`unionByName` combines traffic files by column name. `allowMissingColumns=True` keeps the four extra fields from 20 February and represents them as null for files that did not supply them. No IP addresses are inferred or recovered. Missing-value treatment comes later.

Rename columns to lowercase with underscores for easier Spark SQL. This changes column names only.


In [12]:
traffic_frames = []
for path in traffic_files:
    frame = (spark.read.schema(traffic_schemas[path.name])
             .option("header", True).option("enforceSchema", False)
             .option("mode", "FAILFAST").option("escape", '"')
             .csv(str(sample_folder / path.name)))
    traffic_frames.append(frame)

traffic_df = traffic_frames[0]
for frame in traffic_frames[1:]:
    traffic_df = traffic_df.unionByName(frame, allowMissingColumns=True)

iana_df = (spark.read.schema(IANA_SCHEMA)
           .option("header", True).option("enforceSchema", False)
           .option("mode", "FAILFAST").option("escape", '"')
           .option("multiLine", True).csv(str(registry_file)))

traffic_df = traffic_df.toDF(*[name.lower().replace(" ", "_").replace("/", "_") for name in traffic_df.columns])
iana_df = iana_df.toDF(*[name.lower().replace(" ", "_") for name in iana_df.columns])

traffic_df.select("dst_port", "protocol", "flow_duration", "label").printSchema()
traffic_df.select("dst_port", "protocol", "flow_duration", "label").show(5)


root
 |-- dst_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- label: string (nullable = true)



+--------+--------+-------------+------+
|dst_port|protocol|flow_duration| label|
+--------+--------+-------------+------+
|     443|       6|       141385|Benign|
|   49684|       6|          281|Benign|
|     443|       6|       279824|Benign|
|     443|       6|          132|Benign|
|     443|       6|       274016|Benign|
+--------+--------+-------------+------+
only showing top 5 rows


### 3C - Write Parquet, read it back, and check

Parquet stores column types. Rerunning this cell replaces only these test outputs under `data/parquet/original/sample/`.

We check the expected row counts and column types. `exceptAll` checks for changed or missing rows, including duplicate occurrences. Equal counts plus no missing rows confirm that the contents were preserved. No rows are cleaned or removed here.

In [13]:
parquet_folder = ROOT / "data" / "parquet" / "original" / "sample"
traffic_df.write.mode("overwrite").parquet(str(parquet_folder / "traffic"))
iana_df.write.mode("overwrite").parquet(str(parquet_folder / "iana"))

traffic_parquet = spark.read.parquet(str(parquet_folder / "traffic"))
iana_parquet = spark.read.parquet(str(parquet_folder / "iana"))

for name, original, restored, expected in [
    ("Traffic", traffic_df, traffic_parquet, len(sample_rows)),
    ("IANA", iana_df, iana_parquet, len(registry_rows)),
]:
    actual = restored.count()
    assert actual == expected, f"{name}: row count changed"
    assert original.dtypes == restored.dtypes, f"{name}: column types changed"
    assert original.exceptAll(restored).count() == 0, f"{name}: row contents changed"
    print(f"{name}: {actual} rows; types and contents preserved.")

print("Step 3 passed. Parquet saved in:", parquet_folder)

Traffic: 5000 rows; types and contents preserved.


IANA: 14536 rows; types and contents preserved.
Step 3 passed. Parquet saved in: /home/jovyan/work/data/parquet/original/sample


**Step 3 complete.** The traffic sample and IANA registry have been read with explicit schemas and saved as typed Parquet. Counts, types and contents were checked after reading them back.

Next is Step 4: inspect data quality before deciding cleaning rules. Full traffic ingestion and research analysis are still pending.